# Comparing Keras and PyTorch CNN Models


In [ ]:
import os
import urllib.request
import tarfile

data_dir = "."
dataset_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/4Z1fwRR295-1O3PMQBH6Dg/images-dataSAT.tar"

tar_path = os.path.join(data_dir, "images-dataSAT.tar")

if not os.path.exists(os.path.join(data_dir, "images_dataSAT")):
    print("Downloading dataset...")
    urllib.request.urlretrieve(dataset_url, tar_path)
    print("Extracting...")
    with tarfile.open(tar_path) as tar:
        tar.extractall(data_dir)
    print("Done.")
else:
    print("Dataset already exists, skipping download.")


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
import torch.nn.functional as F



In [ ]:
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Flatten, Dropout, BatchNormalization
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.initializers import HeUniform
from tensorflow.keras.callbacks import ModelCheckpoint

gpu_list = tf.config.list_physical_devices("GPU")
device = "gpu" if gpu_list != [] else "cpu"
print(f"Device available for training: {device}")


In [ ]:
from sklearn.metrics import (accuracy_score,
                             precision_score,
                             recall_score,
                             f1_score,
                             roc_curve,
                             roc_auc_score,
                             log_loss,
                             classification_report,
                             confusion_matrix,
                             ConfusionMatrixDisplay,
                            )
from sklearn.preprocessing import label_binarize

def model_metrics(y_true, y_pred, y_prob, class_labels):
    y_prob_arr = np.array(y_prob)
    y_prob_pos = y_prob_arr.flatten() if y_prob_arr.ndim == 1 or y_prob_arr.shape[1] == 1 else y_prob_arr[:, 1]
    metrics = {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "Loss": log_loss(y_true, y_prob_arr),
        "F1 Score": f1_score(y_true, y_pred),
        "ROC-AUC": roc_auc_score(y_true, y_prob_pos),
        "Confusion Matrix": confusion_matrix(y_true, y_pred),
        "Classification Report": classification_report(y_true, y_pred, target_names=class_labels, digits=4),
        "Class labels": class_labels
    }
    return metrics

def print_metrics(y_true, y_pred, y_prob, class_labels, model_name="Model"):
    m = model_metrics(y_true, y_pred, y_prob, class_labels)
    print(f"\n===== {model_name} Metrics =====")
    for k in ["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC", "Loss"]:
        print(f"{k}: {m[k]:.4f}")
    print("\nConfusion Matrix:")
    print(m["Confusion Matrix"])
    print("\nClassification Report:")
    print(m["Classification Report"])



In [ ]:
def download_model(url, model_path):
    if not os.path.exists(model_path):
        try:
            print(f"Downloading from {url}...")
            urllib.request.urlretrieve(url, model_path)
            print(f"Successfully downloaded '{model_path}'.")
        except Exception as e:
            print(f"Download error: {e}")
    else:
        print(f"Model file already downloaded at: {model_path}")


In [ ]:
import urllib.request

data_dir = "."

keras_model_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/U-uPeyCyOQYh0GrZPGsqoQ/ai-capstone-keras-best-model-model.keras"
keras_model_name = "ai-capstone-keras-best-model-model_downloaded.keras"
keras_model_path = os.path.join(data_dir, keras_model_name)

pytorch_state_dict_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/8J2QEyQqD8x9zjrlnv6N7g/ai-capstone-pytorch-best-model-20250713.pth"
pytorch_state_dict_name = "ai_capstone_pytorch_best_model_state_dict_downloaded.pth"
pytorch_state_dict_path = os.path.join(data_dir, pytorch_state_dict_name)

download_model(keras_model_url, keras_model_path)
download_model(pytorch_state_dict_url, pytorch_state_dict_path)


In [ ]:
dataset_path = os.path.join(data_dir, "images_dataSAT")
print(dataset_path)

img_w, img_h = 64, 64
n_channels = 3
batch_size = 128
num_classes = 2

agri_class_labels = ["non-agri", "agri"]


In [ ]:
datagen = ImageDataGenerator(rescale=1./255)
prediction_generator = datagen.flow_from_directory(
    dataset_path,
    target_size=(img_w, img_h),
    batch_size=batch_size,
    class_mode="binary",
    shuffle=False
)

keras_model = tf.keras.models.load_model(keras_model_path)

steps = int(np.ceil(prediction_generator.samples / prediction_generator.batch_size))
batch_size_local = int(prediction_generator.batch_size)
print(f"Number of Steps: {steps} with batch size: {batch_size_local}")

all_preds_keras = []
all_probs_keras = []
all_labels_keras = []

for step in tqdm(range(steps), desc="Steps"):
    images, labels = next(prediction_generator)
    preds = keras_model.predict(images, verbose=0)
    all_probs_keras.extend(preds)
    preds = (preds > 0.5).astype(int).flatten()
    all_preds_keras.extend(preds)
    all_labels_keras.extend(labels)


### Keras CNN model evaluation metrics

In [ ]:
print_metrics(y_true=all_labels_keras,
              y_pred=all_preds_keras,
              y_prob=all_probs_keras,
              class_labels=agri_class_labels,
              model_name="Keras Model"
             )


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Processing inference on {device}")

train_transform = transforms.Compose([
    transforms.Resize((img_w, img_h)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
full_dataset = datasets.ImageFolder(dataset_path, transform=train_transform)
test_loader = DataLoader(full_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

num_classes = 2
model = nn.Sequential(
    nn.Conv2d(3, 32, 5, padding=2), nn.ReLU(),
    nn.MaxPool2d(2), nn.BatchNorm2d(32),
    nn.Conv2d(32, 64, 5, padding=2), nn.ReLU(), nn.MaxPool2d(2), nn.BatchNorm2d(64),
    nn.Conv2d(64, 128, 5, padding=2), nn.ReLU(), nn.MaxPool2d(2), nn.BatchNorm2d(128),
    nn.Conv2d(128, 256, 5, padding=2), nn.ReLU(), nn.MaxPool2d(2), nn.BatchNorm2d(256),
    nn.Conv2d(256, 512, 5, padding=2), nn.ReLU(), nn.MaxPool2d(2), nn.BatchNorm2d(512),
    nn.Conv2d(512, 1024, 5, padding=2), nn.ReLU(), nn.MaxPool2d(2), nn.BatchNorm2d(1024),
    nn.AdaptiveAvgPool2d(1), nn.Flatten(),
    nn.Linear(1024, 2048), nn.ReLU(), nn.BatchNorm1d(2048), nn.Dropout(0.4),
    nn.Linear(2048, num_classes)
).to(device)

print("Created model, now loading the weights...")
state_dict = torch.load(pytorch_state_dict_path, map_location=device, weights_only=False)
model.load_state_dict(state_dict)
model.eval()
print("Model weights loaded.")

all_preds_pytorch = []
all_probs_pytorch = []
all_labels_pytorch = []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Inference"):
        images = images.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1).cpu().numpy()
        preds = np.argmax(probs, axis=1)
        all_probs_pytorch.extend(probs)
        all_preds_pytorch.extend(preds)
        all_labels_pytorch.extend(labels.numpy())

print("Inference complete.")


### PyTorch CNN model evaluation metrics

In [ ]:
print_metrics(y_true=all_labels_pytorch,
              y_pred=all_preds_pytorch,
              y_prob=all_probs_pytorch,
              class_labels=agri_class_labels,
              model_name="PyTorch Model"
             )


In [ ]:
def plot_roc(y_true, y_prob, model_name):
    y_prob_arr = np.array(y_prob)
    n_classes = y_prob_arr.shape[1] if y_prob_arr.ndim > 1 else 1
    if n_classes == 1 or n_classes == 2:
        y_prob_pos = y_prob_arr.flatten() if y_prob_arr.ndim == 1 or y_prob_arr.shape[1] == 1 else y_prob_arr[:, 1]
        fpr, tpr, _ = roc_curve(y_true, y_prob_pos)
        auc = roc_auc_score(y_true, y_prob_pos)
        plt.plot(fpr, tpr, label=f'{model_name} (AUC = {auc:.2f})')
    else:
        y_true_bin = label_binarize(y_true, classes=np.arange(n_classes))
        for i in range(n_classes):
            fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_prob_arr[:, i])
            auc = roc_auc_score(y_true_bin[:, i], y_prob_arr[:, i])
            plt.plot(fpr, tpr, label=f'{model_name} class {i} (AUC = {auc:.2f})')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curve')
    plt.legend()


In [ ]:
plot_roc(np.array(all_labels_keras), np.array(all_probs_keras), "Keras Model")
plt.show()
plot_roc(np.array(all_labels_pytorch), np.array(all_probs_pytorch), "PyTorch Model")
plt.show()

In [ ]:
# compute metrics for Keras
metrics_keras = model_metrics(all_labels_keras, all_preds_keras, all_probs_keras, agri_class_labels)

# compute metrics for PyTorch
metrics_pytorch = model_metrics(all_labels_pytorch, all_preds_pytorch, all_probs_pytorch, agri_class_labels)

# side-by-side comparison table
print("{:<18} {:<15} {:<15}".format('\033[1m'+ 'Metric' + '\033[0m',
                                    'Keras Model', 
                                    'PyTorch Model'))

metrics_list = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC']

for k in metrics_list:
    print("{:<18} {:<15.4f} {:<15.4f}".format('\033[1m'+k+'\033[0m',
                                              metrics_keras[k],
                                              metrics_pytorch[k]))

### Metric analysis

The metrics for the pre-trained Keras and PyTorch models for evaluating the provided dataset are:

- **Accuracy**
    1. Keras: 0.9925
    2. PyTorch: 0.9988
    
    ===> Both models achieve exceptional accuracy, but the **PyTorch model makes fewer mistakes**.

- **Precision**
    1. Keras: 1.0000
    2. PyTorch: 0.9983

    ===> The **Keras** model perfectly **avoids false positives**, whereas the PyTorch model is slightly less perfect but still excellent.

- **Recall**
    1. Keras: 0.9850
    2. PyTorch: 0.9993
    
    ===> The **PyTorch** model is marginally better at **identifying all true positives**, capturing nearly all actual positive cases, while the Keras model misses a few.

- **F1 Score**
    1. PyTorch: 0.9988
    2. Keras: 0.9924
    
    ===> The F1 score, which balances precision and recall, favors the **PyTorch** model thanks to its **stronger recall**.

- **ROC-AUC**
    1. Keras: 1.0000
    2. PyTorch: 1.0000
    
    ===> Both models reach maximum possible **discrimination between classes**, indicating outstanding capability for binary classification.


### **Model comparison: Key insights**


**PyTorch model strengths**

 - Achieves the highest scores in accuracy, recall, and F1, indicating extremely robust overall performance and near-perfect classification of positive cases
- ROC-AUC of 1.0 shows perfect class separability


**Keras model strengths**

- Displays almost perfect precision every positive prediction made is correct
- Also achieves perfect ROC-AUC, indicating outstanding discrimination ability


**Common strength**

- Both models deliver flawless ROC-AUC, suggesting both are highly effective for this classification task


**Recommendations**

Based on the scores from the uploaded pre-trained models:

- The PyTorch model is preferable for applications where missing any positive instances is costly (higher recall)
- The Keras model is optimal for scenarios where making any false positive error is unacceptable (higher precision).


**Next**

- Analyze the confusion matrices to investigate the errors.
- Monitor real-world performance, as even marginal differences can become important in high-impact applications. 


**Summary**

Both models excel in all evaluated metrics and would be highly reliable in production. The PyTorch model demonstrates a modest edge in recall and F1 score, while the Keras model maximizes precision. The choice between models should ultimately reflect the specific requirements and risk tolerance of your use case.

